# scGPT Pretrain (Colab Clean Workflow)

This notebook is a cleaned, linear workflow for pretraining scGPT:
1. Environment setup + scGPT import
2. One config cell (smoke test or real run)
3. Data discovery and sanity check
4. Vocab + shard manifest
5. Lazy dataset + dataloader
6. Training + checkpoint saving

Tip: run once with `SMOKE_TEST=True`, then switch to real data.


In [ ]:
# 1) Colab environment setup (connect GPU first)
# Install runtime dependencies used by this pretrain pipeline.
!pip -q install --upgrade pip
!pip -q install scanpy anndata scipy pandas numpy scikit-learn tqdm psutil datasets

# Clone scGPT source code into Colab local disk.
!rm -rf /content/scGPT
!git clone https://github.com/bowang-lab/scGPT.git /content/scGPT

# Overwrite scgpt/__init__.py with a safe variant where trainer import is optional.
# This keeps core pretrain imports working without torchtext.
from pathlib import Path

init_path = Path('/content/scGPT/scgpt/__init__.py')
safe_init = """__version__ = "0.2.4"
import logging
import sys

logger = logging.getLogger("scGPT")
if not logger.hasHandlers() or len(logger.handlers) == 0:
    logger.propagate = False
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter(
        "%(name)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

from . import model, tokenizer, scbank, utils, tasks
from .data_collator import DataCollator
from .data_sampler import SubsetsBatchSampler

try:
    from .trainer import (
        prepare_data,
        prepare_dataloader,
        train,
        define_wandb_metrcis,
        evaluate,
        eval_testdata,
        test,
    )
except Exception:
    # Keep core package importable when optional trainer-time deps are missing.
    pass
"""
init_path.write_text(safe_init, encoding='utf-8')
print('Patched scgpt/__init__.py (optional trainer import)')

# Add the cloned repo to import path.
import sys
if "/content/scGPT" not in sys.path:
    sys.path.insert(0, "/content/scGPT")

# Quick runtime check.
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# 2) Mount Google Drive
# We store input h5ad files and output checkpoints in Drive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3) Unified config (edit only this cell)
import os

CFG = {
    # Paths
    "DATA_DIR": "/content/drive/MyDrive/scgpt_real_data",
    "OUTPUT_DIR": "/content/drive/MyDrive/scgpt_runs/pretrain_clean_v1",
    "FILE_EXT": ".h5ad",
    # Set to "counts" if your raw counts are in adata.layers['counts'].
    # Keep None to use adata.X.
    "LAYER_KEY": None,

    # Run mode
    "SMOKE_TEST": False,
    # None means all files. Set 2/5 for quick debugging.
    "MAX_FILES": None,
    "MAX_CELLS_TOTAL": 100000,
    "VAL_FRACTION": 0.01,

    # Tokens and masking values
    "PAD_TOKEN": "<pad>",
    "CLS_TOKEN": "<cls>",
    "PAD_VALUE": 0,
    "MASK_VALUE": -1,

    # Sequence and masking behavior
    "MAX_LEN": 2048,
    "N_BINS": 51,
    "MLM_PROB": 0.15,

    # Training hyperparameters
    "BATCH_SIZE": 4,
    "GRAD_ACCUM_STEPS": 8,
    "EPOCHS": 3,
    "LR": 1e-4,
    "WEIGHT_DECAY": 1e-4,
    "CLIP_GRAD": 1.0,
    "NUM_WORKERS": 0,
    "USE_AMP": True,

    # Model size
    "EMB_SIZE": 256,
    "N_HEAD": 8,
    "N_LAYERS": 4,
    "FFN_HID": 512,
    "DROPOUT": 0.2,
    "USE_FAST_TRANSFORMER": False,

    # Misc
    "SEED": 42,
    # Resume from checkpoint path, e.g. "/content/drive/.../last.pt"
    "RESUME_CKPT": None,
}

if CFG["SMOKE_TEST"]:
    # Faster defaults for quick end-to-end validation.
    CFG.update({
        "MAX_FILES": 2,
        "MAX_CELLS_TOTAL": 5000,
        "VAL_FRACTION": 0.05,
        "MAX_LEN": 1200,
        "BATCH_SIZE": 8,
        "GRAD_ACCUM_STEPS": 2,
        "EPOCHS": 1,
    })

CFG["SPECIAL_TOKENS"] = [CFG["PAD_TOKEN"], CFG["CLS_TOKEN"]]
os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)

print("==== CONFIG ====")
for k, v in CFG.items():
    print(f"{k}: {v}")


In [ ]:
# 4) Imports + helper functions
import gc
import json
import random
from bisect import bisect_right
from contextlib import nullcontext
from pathlib import Path

import anndata as ad
import numpy as np
import scipy.sparse as sp
import torch
from torch.utils.data import Dataset, DataLoader, Subset, random_split, get_worker_info
from tqdm.auto import tqdm

from scgpt.data_collator import DataCollator
from scgpt.loss import masked_mse_loss
from scgpt.model.model import TransformerModel


def seed_everything(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_data_files(data_dir: str, ext: str = ".h5ad", max_files=None):
    """Recursively collect input files under DATA_DIR."""
    files = sorted(str(p) for p in Path(data_dir).rglob(f"*{ext}"))
    if max_files is not None:
        files = files[:max_files]
    if len(files) == 0:
        raise FileNotFoundError(f"No files found under {data_dir} with suffix {ext}")
    return files


def inspect_first_file(fp: str):
    """Print minimal schema info from one file for sanity check."""
    a = ad.read_h5ad(fp, backed="r")
    try:
        print("sample file:", fp)
        print("n_obs:", int(a.n_obs), "n_vars:", int(a.n_vars))
        print("layers:", list(a.layers.keys()))
        print("first 10 genes:", list(map(str, a.var_names[:10])))
    finally:
        if hasattr(a, "file") and a.file is not None:
            a.file.close()


seed_everything(CFG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = bool(CFG["USE_AMP"] and device.type == "cuda")

# Enable TF32 on NVIDIA Ampere+ for faster matmul/convolution.
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", device)
print("amp_enabled:", amp_enabled)


In [ ]:
# 5) Discover data files + quick check
# Build the file list from DATA_DIR.
data_files = find_data_files(
    data_dir=CFG["DATA_DIR"],
    ext=CFG["FILE_EXT"],
    max_files=CFG["MAX_FILES"],
)

print("num files:", len(data_files))
print("first 5 files:")
for p in data_files[:5]:
    print(" ", p)

# Confirm file structure before expensive preprocessing.
inspect_first_file(data_files[0])


In [ ]:
# 6) Build / load vocab
class SimpleGeneVocab:
    """A lightweight token-to-id mapping for genes + special tokens."""

    def __init__(self, genes=None, specials=("<pad>", "<cls>")):
        genes = [] if genes is None else genes
        uniq = []
        seen = set()

        # Put special tokens at the beginning in a deterministic order.
        for tok in specials:
            tok = str(tok)
            if tok not in seen:
                uniq.append(tok)
                seen.add(tok)

        # Add gene symbols/names while preserving first occurrence order.
        for g in genes:
            g = str(g)
            if g not in seen:
                uniq.append(g)
                seen.add(g)

        self.itos = uniq
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __contains__(self, token):
        return str(token) in self.stoi

    def __getitem__(self, token):
        return self.stoi[str(token)]

    def get_stoi(self):
        return self.stoi

    def get_itos(self):
        return self.itos

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.stoi, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            stoi = json.load(f)

        obj = cls([])
        obj.stoi = {str(k): int(v) for k, v in stoi.items()}
        obj.itos = [None] * len(obj.stoi)
        for tok, idx in obj.stoi.items():
            obj.itos[idx] = tok
        return obj


def build_vocab(file_list, specials):
    """Scan all files and collect a global gene vocabulary."""
    all_genes = set()
    for fp in tqdm(file_list, desc="Scanning genes"):
        a = ad.read_h5ad(fp, backed="r")
        try:
            all_genes.update(map(str, a.var_names))
        finally:
            if hasattr(a, "file") and a.file is not None:
                a.file.close()
            del a
            gc.collect()

    gene_list = sorted(all_genes)
    return SimpleGeneVocab(gene_list, specials=specials)


VOCAB_PATH = os.path.join(CFG["OUTPUT_DIR"], "vocab.json")
if os.path.exists(VOCAB_PATH):
    # Reuse existing vocab for stable token ids across runs.
    vocab = SimpleGeneVocab.load(VOCAB_PATH)
    print("loaded vocab from:", VOCAB_PATH)
else:
    vocab = build_vocab(data_files, specials=CFG["SPECIAL_TOKENS"])
    vocab.save(VOCAB_PATH)
    print("built and saved vocab to:", VOCAB_PATH)

assert CFG["PAD_TOKEN"] in vocab, "PAD token missing in vocab"
assert CFG["CLS_TOKEN"] in vocab, "CLS token missing in vocab"

print("vocab size:", len(vocab))
print("PAD id:", vocab[CFG["PAD_TOKEN"]], "CLS id:", vocab[CFG["CLS_TOKEN"]])


In [ ]:
# 7) Build shard manifest (map var_names to vocab ids)
def build_shard_manifest(file_list, vocab):
    """Prepare per-file metadata so __getitem__ can stay lightweight."""
    shard_infos = []
    cumulative = []
    total = 0
    stoi = vocab.get_stoi()

    for fp in tqdm(file_list, desc="Building shard manifest"):
        a = ad.read_h5ad(fp, backed="r")
        try:
            # Convert each file's var_names into global vocab ids.
            genes = np.array(list(map(str, a.var_names)), dtype=object)
            gene_ids = np.array([stoi[g] if g in stoi else -1 for g in genes], dtype=np.int64)

            n_obs = int(a.n_obs)
            shard_infos.append(
                {
                    "path": fp,
                    "n_obs": n_obs,
                    "n_vars": int(a.n_vars),
                    "gene_ids": gene_ids,
                }
            )

            total += n_obs
            cumulative.append(total)
        finally:
            if hasattr(a, "file") and a.file is not None:
                a.file.close()
            del a
            gc.collect()

    return shard_infos, cumulative, total


shard_infos, cumulative_sizes, total_cells = build_shard_manifest(data_files, vocab)
print("total cells across files:", total_cells)
if total_cells == 0:
    raise ValueError("Total cell count is zero. Please check input files.")


In [ ]:
# 8) Lazy Dataset + train/val split + DataLoader
class ScGPTLazyPretrainDataset(Dataset):
    """
    Lazily reads one cell at a time from backed h5ad files.
    Each item contains non-zero genes plus a leading CLS token.
    """

    def __init__(self, shard_infos, cumulative_sizes, vocab, cls_token="<cls>", layer_key=None):
        self.shard_infos = shard_infos
        self.cumulative_sizes = cumulative_sizes
        self.vocab = vocab
        self.cls_token = cls_token
        self.layer_key = layer_key
        # Cache open backed readers by (worker_id, shard_id).
        self._readers = {}

    def __len__(self):
        return 0 if len(self.cumulative_sizes) == 0 else self.cumulative_sizes[-1]

    def _locate_index(self, idx):
        """Map global index -> (shard index, local row index)."""
        shard_idx = bisect_right(self.cumulative_sizes, idx)
        prev = 0 if shard_idx == 0 else self.cumulative_sizes[shard_idx - 1]
        local_idx = idx - prev
        return shard_idx, local_idx

    def _reader_key(self, shard_idx):
        wi = get_worker_info()
        worker_id = -1 if wi is None else wi.id
        return (worker_id, shard_idx)

    def _get_reader(self, shard_idx):
        """Open backed file on demand and reuse within each worker."""
        key = self._reader_key(shard_idx)
        if key not in self._readers:
            fp = self.shard_infos[shard_idx]["path"]
            self._readers[key] = ad.read_h5ad(fp, backed="r")
        return self._readers[key]

    def _get_row(self, reader, local_idx):
        """Fetch one cell vector from X or from a specified layer."""
        if self.layer_key is None:
            return reader.X[local_idx]
        return reader.layers[self.layer_key][local_idx]

    def __getitem__(self, idx):
        shard_idx, local_idx = self._locate_index(idx)
        info = self.shard_infos[shard_idx]
        reader = self._get_reader(shard_idx)

        row = self._get_row(reader, local_idx)
        gene_ids_all = info["gene_ids"]

        # Keep only non-zero entries to reduce sequence length.
        if sp.issparse(row):
            row = row.tocsr()
            nz_idx = row.indices
            nz_val = row.data.astype(np.float32)
        else:
            row = np.asarray(row).reshape(-1)
            nz_idx = np.flatnonzero(row)
            nz_val = row[nz_idx].astype(np.float32)

        if len(nz_idx) == 0:
            # Edge case: all-zero cell -> CLS only.
            genes = np.array([self.vocab[self.cls_token]], dtype=np.int64)
            exprs = np.array([0.0], dtype=np.float32)
        else:
            mapped_gene_ids = gene_ids_all[nz_idx]
            keep = mapped_gene_ids >= 0
            mapped_gene_ids = mapped_gene_ids[keep]
            nz_val = nz_val[keep]

            # Prepend CLS so keep_first_n_tokens=1 stays valid in DataCollator.
            genes = np.concatenate([[self.vocab[self.cls_token]], mapped_gene_ids]).astype(np.int64)
            exprs = np.concatenate([[0.0], nz_val]).astype(np.float32)

        return {
            "id": torch.tensor(idx, dtype=torch.long),
            "genes": torch.tensor(genes, dtype=torch.long),
            "expressions": torch.tensor(exprs, dtype=torch.float32),
        }

    def close(self):
        """Best-effort cleanup for backed file handles."""
        for reader in self._readers.values():
            try:
                if hasattr(reader, "file") and reader.file is not None:
                    reader.file.close()
            except Exception:
                pass
        self._readers.clear()


full_dataset = ScGPTLazyPretrainDataset(
    shard_infos=shard_infos,
    cumulative_sizes=cumulative_sizes,
    vocab=vocab,
    cls_token=CFG["CLS_TOKEN"],
    layer_key=CFG["LAYER_KEY"],
)
print("full dataset size:", len(full_dataset))

# Optional down-sampling for faster iteration/debugging.
if CFG["MAX_CELLS_TOTAL"] is not None and CFG["MAX_CELLS_TOTAL"] < len(full_dataset):
    g = torch.Generator().manual_seed(CFG["SEED"])
    subset_idx = torch.randperm(len(full_dataset), generator=g)[: CFG["MAX_CELLS_TOTAL"]]
    dataset = Subset(full_dataset, subset_idx.tolist())
else:
    dataset = full_dataset

print("working dataset size:", len(dataset))
if len(dataset) < 2:
    raise ValueError("Need at least 2 cells after filtering/subsampling for train/val split.")

# Train/validation split.
val_size = max(1, int(len(dataset) * CFG["VAL_FRACTION"]))
if val_size >= len(dataset):
    val_size = len(dataset) - 1
train_size = len(dataset) - val_size

split_g = torch.Generator().manual_seed(CFG["SEED"])
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=split_g)
print("train size:", len(train_dataset), "val size:", len(val_dataset))

# scGPT collator does padding, optional binning, and MLM masking.
collator = DataCollator(
    do_padding=True,
    pad_token_id=vocab[CFG["PAD_TOKEN"]],
    pad_value=CFG["PAD_VALUE"],
    do_mlm=True,
    do_binning=True,
    mlm_probability=CFG["MLM_PROB"],
    mask_value=CFG["MASK_VALUE"],
    max_length=CFG["MAX_LEN"],
    sampling=True,
    keep_first_n_tokens=1,
)

pin_memory = device.type == "cuda"
# drop_last avoids tiny last batch instability when batch size is larger than dataset tail.
train_drop_last = len(train_dataset) >= CFG["BATCH_SIZE"]

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=pin_memory,
    collate_fn=collator,
    drop_last=train_drop_last,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=pin_memory,
    collate_fn=collator,
    drop_last=False,
)

# Quick batch shape check.
batch = next(iter(train_loader))
print("batch keys:", batch.keys())
for k, v in batch.items():
    print(k, tuple(v.shape), v.dtype)


In [ ]:
# 9) Build model + optimizer + core functions
model = TransformerModel(
    ntoken=len(vocab),
    d_model=CFG["EMB_SIZE"],
    nhead=CFG["N_HEAD"],
    d_hid=CFG["FFN_HID"],
    nlayers=CFG["N_LAYERS"],
    nlayers_cls=3,
    n_cls=1,
    vocab=vocab,
    dropout=CFG["DROPOUT"],
    pad_token=CFG["PAD_TOKEN"],
    pad_value=CFG["PAD_VALUE"],
    do_mvc=False,
    do_dab=False,
    use_batch_labels=False,
    num_batch_labels=None,
    domain_spec_batchnorm=False,
    input_emb_style="continuous",
    n_input_bins=CFG["N_BINS"],
    cell_emb_style="cls",
    mvc_decoder_style="inner product",
    ecs_threshold=0.3,
    explicit_zero_prob=False,
    use_fast_transformer=CFG["USE_FAST_TRANSFORMER"],
    fast_transformer_backend="flash",
    pre_norm=False,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params/1e6:.2f} M")

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

# Use new API if available; fallback keeps compatibility.
try:
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)


def move_batch_to_device(batch, device):
    """Move all batch tensors to target device."""
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


def autocast_context():
    """Return autocast context only when AMP is enabled."""
    if amp_enabled:
        return torch.autocast(device_type="cuda")
    return nullcontext()


def forward_and_loss(model, batch):
    """Forward pass + masked MSE loss for the MLM regression target."""
    genes = batch["gene"]
    target_values = batch["expr"].float()
    masked_values = batch["masked_expr"].float()

    src_key_padding_mask = genes.eq(vocab[CFG["PAD_TOKEN"]])
    mlm_mask = masked_values.eq(CFG["MASK_VALUE"])

    output = model(
        src=genes,
        values=masked_values,
        src_key_padding_mask=src_key_padding_mask,
        batch_labels=None,
        CLS=False,
        CCE=False,
        MVC=False,
        ECS=False,
    )
    loss = masked_mse_loss(output["mlm_output"], target_values, mlm_mask)
    return loss, output

# Quick forward check before full training.
quick_batch = move_batch_to_device(next(iter(train_loader)), device)
with autocast_context():
    quick_loss, quick_output = forward_and_loss(model, quick_batch)

print("quick forward loss:", float(quick_loss.item()))
print("mlm_output shape:", tuple(quick_output["mlm_output"].shape))


In [ ]:
# 10) Training loop + checkpoint
best_val = float("inf")
start_epoch = 1
history = {"train_loss": [], "val_loss": []}

# Optional resume from last checkpoint.
if CFG["RESUME_CKPT"] is not None and os.path.exists(CFG["RESUME_CKPT"]):
    ckpt = torch.load(CFG["RESUME_CKPT"], map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    history = ckpt.get("history", history)
    best_val = float(np.min(history["val_loss"])) if len(history["val_loss"]) > 0 else best_val
    start_epoch = int(ckpt.get("epoch", 0)) + 1
    print(f"Resumed from {CFG['RESUME_CKPT']} at epoch {start_epoch}")


@torch.no_grad()
def evaluate(model, loader):
    """Validation pass."""
    model.eval()
    losses = []
    for batch in tqdm(loader, desc="valid", leave=False):
        batch = move_batch_to_device(batch, device)
        with autocast_context():
            loss, _ = forward_and_loss(model, batch)
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")


def train_one_epoch(model, loader, optimizer, scaler):
    """One epoch with gradient accumulation + mixed precision."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_losses = []

    pbar = tqdm(loader, desc="train", leave=False)
    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch, device)

        with autocast_context():
            loss, _ = forward_and_loss(model, batch)
            loss = loss / CFG["GRAD_ACCUM_STEPS"]

        scaler.scale(loss).backward()

        # Update optimizer every GRAD_ACCUM_STEPS mini-batches.
        if step % CFG["GRAD_ACCUM_STEPS"] == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP_GRAD"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_losses.append(loss.item() * CFG["GRAD_ACCUM_STEPS"])
        pbar.set_postfix(train_loss=f"{np.mean(running_losses[-20:]):.4f}")

    return float(np.mean(running_losses))


for epoch in range(start_epoch, CFG["EPOCHS"] + 1):
    print(f"\nEpoch {epoch}/{CFG['EPOCHS']}")

    train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss = evaluate(model, val_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(f"train_loss = {train_loss:.6f}")
    print(f"val_loss   = {val_loss:.6f}")

    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "history": history,
        "config": CFG,
    }

    last_path = os.path.join(CFG["OUTPUT_DIR"], "last.pt")
    best_path = os.path.join(CFG["OUTPUT_DIR"], "best.pt")
    hist_path = os.path.join(CFG["OUTPUT_DIR"], "history.json")

    # Always save latest checkpoint.
    torch.save(ckpt, last_path)
    with open(hist_path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    # Save best checkpoint by validation loss.
    if val_loss < best_val:
        best_val = val_loss
        torch.save(ckpt, best_path)
        print("saved best checkpoint")

print("\nTraining done.")
print("Artifacts in:", CFG["OUTPUT_DIR"])

# Close backed readers to avoid stale file handles.
if isinstance(full_dataset, ScGPTLazyPretrainDataset):
    full_dataset.close()


## How To Use

- First run once with `SMOKE_TEST=True` to validate the full pipeline.
- Then set `SMOKE_TEST=False` and confirm:
  - `DATA_DIR` points to your real `.h5ad` directory.
  - `LAYER_KEY` is set correctly (e.g., `"counts"` if needed).
  - `MAX_LEN` / `BATCH_SIZE` fit your GPU memory.
- If interrupted, set `RESUME_CKPT` to `last.pt` and continue training.
